In [45]:
import numpy as np
import pandas as pd
import os

H = 21

MODEL_ORDER = ["EWMA", "DCC", "HAR"]

PROXY_DIR = "../proxy"
RESULTS_DIR = "../results"
MODELS_DIR = "../models"

In [46]:
def load_cov_csv(path):
    return pd.read_csv(
        path,
        parse_dates=["Date"]
    ).set_index("Date").sort_index()


# -------------------------
# Load proxy
# -------------------------

proxy_path = os.path.join(PROXY_DIR, f"realized_cov_h{H}.csv")

proxy = load_cov_csv(proxy_path)

print(f"Loaded proxy (h={H})")
print(f"File: {proxy_path}")
print("Shape:", proxy.shape)
print("Date range:", proxy.index.min().date(), "->", proxy.index.max().date())
print()


# -------------------------
# Load models
# -------------------------

models = {}

for model in MODEL_ORDER:

    if model == "EWMA":
        path = os.path.join(
            MODELS_DIR,
            "ewma_cov_lambda094.csv"
        )
    else:
        path = os.path.join(
            RESULTS_DIR,
            f"{model.lower()}_cov_forecast_h{H}.csv"
        )

    if os.path.exists(path):

        models[model] = load_cov_csv(path)

        df = models[model]

        print(f"Loaded {model}")
        print(f"File: {path}")
        print("Shape:", df.shape)
        print("Date range:", df.index.min().date(), "->", df.index.max().date())
        print()

    else:
        print(f"[skip] Missing file for {model}: {path}")

Loaded proxy (h=21)
File: ../proxy/realized_cov_h21.csv
Shape: (2184, 64)
Date range: 2017-03-28 -> 2025-12-02

Loaded EWMA
File: ../models/ewma_cov_lambda094.csv
Shape: (1255, 64)
Date range: 2021-01-04 -> 2025-12-31

Loaded DCC
File: ../results/dcc_cov_forecast_h21.csv
Shape: (1255, 64)
Date range: 2021-01-04 -> 2025-12-31

Loaded HAR
File: ../results/har_cov_forecast_h21.csv
Shape: (1235, 64)
Date range: 2021-01-04 -> 2025-12-02



In [47]:
# -------------------------
# Align evaluation sample
# -------------------------

common_index = proxy.index

for df in models.values():
    common_index = common_index.intersection(df.index)

proxy = proxy.loc[common_index]

for name in models:
    models[name] = models[name].loc[common_index]


print("Aligned evaluation sample")
print("Start:", common_index.min().date())
print("End  :", common_index.max().date())
print("Observations:", len(common_index))
print()

print("Proxy", proxy.shape)

for name, df in models.items():
    print(name, df.shape)

Aligned evaluation sample
Start: 2021-01-04
End  : 2025-12-02
Observations: 1235

Proxy (1235, 64)
EWMA (1235, 64)
DCC (1235, 64)
HAR (1235, 64)


In [48]:
n_assets = int(np.sqrt(proxy.shape[1]))
print("Number of assets:", n_assets)

Number of assets: 8


In [49]:
# ============================================================
# Matrix utilities
# ============================================================

def row_to_matrix(row, n_assets):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat


def make_spd(mat, eps=1e-8):
    mat = 0.5 * (mat + mat.T)

    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals[eigvals < eps] = eps

    mat_spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    mat_spd = 0.5 * (mat_spd + mat_spd.T)

    return mat_spd


# ============================================================
# Covariance loss functions
# ============================================================

def qlike_loss(proxy_mat, forecast_mat):

    S = proxy_mat
    H = forecast_mat

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)


def frobenius_loss(proxy_mat, forecast_mat):

    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))


# ============================================================
# Covariance evaluation
# ============================================================

def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates]
    proxy_df = proxy_df.loc[common_dates]

    records = []
    spd_fix_count = 0

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets)

        try:
            qlike = qlike_loss(S_mat, H_mat)

        except ValueError:
            H_mat = make_spd(H_mat)
            qlike = qlike_loss(S_mat, H_mat)
            spd_fix_count += 1

        records.append({
            "Date": dt,
            "QLIKE": qlike,
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "avg_frobenius": losses["FROBENIUS"].mean(),
        "spd_fixes": spd_fix_count
    })

    return losses, summary


# ============================================================
# Variance / Correlation utilities
# ============================================================

def covariance_to_variance_vector(cov_mat):
    return np.diag(cov_mat)


def covariance_to_correlation_matrix(cov_mat):

    cov_mat = 0.5 * (cov_mat + cov_mat.T)

    var = np.diag(cov_mat)
    std = np.sqrt(np.maximum(var, 1e-12))

    corr = cov_mat / np.outer(std, std)

    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)

    return corr


# ============================================================
# Component loss functions
# ============================================================

def variance_mse_loss(proxy_mat, forecast_mat):

    var_proxy = covariance_to_variance_vector(proxy_mat)
    var_forecast = covariance_to_variance_vector(forecast_mat)

    return float(np.mean((var_forecast - var_proxy) ** 2))


def correlation_mse_loss(proxy_mat, forecast_mat):

    corr_proxy = covariance_to_correlation_matrix(proxy_mat)
    corr_forecast = covariance_to_correlation_matrix(forecast_mat)

    n = corr_proxy.shape[0]

    tri_i, tri_j = np.triu_indices(n, k=1)

    diff = corr_forecast[tri_i, tri_j] - corr_proxy[tri_i, tri_j]

    return float(np.mean(diff ** 2))


# ============================================================
# Component evaluation
# ============================================================

def evaluate_components_against_proxy(forecast_df, proxy_df, model_name, n_assets):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates]
    proxy_df = proxy_df.loc[common_dates]

    records = []

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets)

        records.append({
            "Date": dt,
            "VAR_MSE": variance_mse_loss(S_mat, H_mat),
            "CORR_MSE": correlation_mse_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_var_mse": losses["VAR_MSE"].mean(),
        "avg_corr_mse": losses["CORR_MSE"].mean()
    })

    return losses, summary


# ============================================================
# System volatility
# ============================================================

def system_volatility(df, n_assets):

    vol = df.apply(
        lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
        axis=1
    )

    return vol

In [50]:
for model_name in MODEL_ORDER:
    if model_name not in models:
        continue

    forecast_df = models[model_name]

    for dt in forecast_df.index.intersection(proxy.index):
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets)
        sign_h, logdet_h = np.linalg.slogdet(H_mat)

        if sign_h <= 0:
            print(f"{model_name} has non-positive determinant on {dt.date()}")
            break

HAR has non-positive determinant on 2021-04-14


Analysis whole sample 

In [51]:
cov_losses = {}
cov_summaries = []

for model_name in MODEL_ORDER:

    if model_name not in models:
        continue

    forecast_df = models[model_name]

    losses, summary = evaluate_against_proxy(
        forecast_df=forecast_df,
        proxy_df=proxy,
        model_name=model_name,
        n_assets=n_assets
    )

    cov_losses[model_name] = losses
    cov_summaries.append(summary)

cov_summary_table = pd.DataFrame(cov_summaries).set_index("model")
cov_summary_table = cov_summary_table.loc[[m for m in MODEL_ORDER if m in cov_summary_table.index]]

print("Covariance Forecast Performance (Full Sample)")
cov_summary_table

Covariance Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_qlike,avg_frobenius,spd_fixes
model,,,,,,
EWMA,1235,2021-01-04,2025-12-02,5.235795,0.000002,0
DCC,1235,2021-01-04,2025-12-02,4.488577,0.000017,0
HAR,1235,2021-01-04,2025-12-02,9.260705,0.000002,8


In [52]:
component_losses = {}
component_summaries = []

for model_name in MODEL_ORDER:

    if model_name not in models:
        continue

    forecast_df = models[model_name]

    losses, summary = evaluate_components_against_proxy(
        forecast_df=forecast_df,
        proxy_df=proxy,
        model_name=model_name,
        n_assets=n_assets
    )

    component_losses[model_name] = losses
    component_summaries.append(summary)

component_summary_table = pd.DataFrame(component_summaries).set_index("model")
component_summary_table = component_summary_table.loc[[m for m in MODEL_ORDER if m in component_summary_table.index]]

print("Variance and Correlation Forecast Performance (Full Sample)")
component_summary_table

Variance and Correlation Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
model,,,,,
EWMA,1235,2021-01-04,2025-12-02,1.491691e-07,0.080336
DCC,1235,2021-01-04,2025-12-02,2.108982e-06,0.075739
HAR,1235,2021-01-04,2025-12-02,1.826949e-07,0.091442


In [53]:
qlike_compare = pd.DataFrame({
    model_name: cov_losses[model_name]["QLIKE"]
    for model_name in cov_losses
})

frobenius_compare = pd.DataFrame({
    model_name: cov_losses[model_name]["FROBENIUS"]
    for model_name in cov_losses
})

print("Best model by QLIKE share:")
print(qlike_compare.idxmin(axis=1).value_counts(normalize=True))

print("Best model by frobenius share:")
print(frobenius_compare.idxmin(axis=1).value_counts(normalize=True))

Best model by QLIKE share:
DCC     0.656680
EWMA    0.238057
HAR     0.105263
Name: proportion, dtype: float64
Best model by frobenius share:
EWMA    0.391903
HAR     0.338462
DCC     0.269636
Name: proportion, dtype: float64


Analysis regime

In [54]:
proxy_vol = proxy.apply(
    lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
    axis=1
)

proxy_vol.name = "system_volatility"

low_threshold = proxy_vol.quantile(0.25)
high_threshold = proxy_vol.quantile(0.75)

print("Low volatility threshold:", low_threshold)
print("High volatility threshold:", high_threshold)

vol_regime = pd.Series(index=proxy_vol.index, dtype="object")
vol_regime[proxy_vol <= low_threshold] = "low"
vol_regime[proxy_vol >= high_threshold] = "high"
vol_regime[(proxy_vol > low_threshold) & (proxy_vol < high_threshold)] = "mid"

vol_regime.value_counts()

Low volatility threshold: 0.03342160900170007
High volatility threshold: 0.04902303949432711


mid     617
high    309
low     309
Name: count, dtype: int64

In [55]:
for model in cov_losses:
    cov_losses[model]["regime"] = vol_regime.loc[cov_losses[model].index]

for model in component_losses:
    component_losses[model]["regime"] = vol_regime.loc[component_losses[model].index]

In [56]:
qlike_regime_table = pd.DataFrame({
    model: cov_losses[model].groupby("regime")["QLIKE"].mean()
    for model in MODEL_ORDER if model in cov_losses
})

frob_regime_table = pd.DataFrame({
    model: cov_losses[model].groupby("regime")["FROBENIUS"].mean()
    for model in MODEL_ORDER if model in cov_losses
})

# fast regime rekkefølge
regime_order = ["low", "mid", "high"]

qlike_regime_table = qlike_regime_table.loc[regime_order]
frob_regime_table = frob_regime_table.loc[regime_order]

print("Average QLIKE in regimes (lower is better)")
display(qlike_regime_table)

print("\nAverage Frobenius loss in regimes (lower is better)")
display(frob_regime_table)

Average QLIKE in regimes (lower is better)


,EWMA,DCC,HAR
regime,,,
low,4.218570,4.214756,4.976704
mid,4.603506,3.813025,9.880377
high,7.515551,6.111316,12.307370



Average Frobenius loss in regimes (lower is better)


,EWMA,DCC,HAR
regime,,,
low,6.027697e-07,0.000048,4.997600e-07
mid,1.211043e-06,0.000008,1.215071e-06
high,3.213900e-06,0.000006,4.478937e-06


In [57]:
var_regime_table = pd.DataFrame({
    model: component_losses[model].groupby("regime")["VAR_MSE"].mean()
    for model in MODEL_ORDER if model in component_losses
})

corr_regime_table = pd.DataFrame({
    model: component_losses[model].groupby("regime")["CORR_MSE"].mean()
    for model in MODEL_ORDER if model in component_losses
})

var_regime_table = var_regime_table.loc[regime_order]
corr_regime_table = corr_regime_table.loc[regime_order]

print("Variance MSE by volatility regime (lower is better)")
display(var_regime_table)

print("\nCorrelation MSE by volatility regime (lower is better)")
display(corr_regime_table)

Variance MSE by volatility regime (lower is better)


,EWMA,DCC,HAR
regime,,,
low,5.092294e-08,5.976040e-06,4.005794e-08
mid,1.171445e-07,9.234530e-07,1.129261e-07
high,3.113609e-07,6.091462e-07,4.646436e-07



Correlation MSE by volatility regime (lower is better)


,EWMA,DCC,HAR
regime,,,
low,0.073199,0.075213,0.083123
mid,0.073534,0.065470,0.085478
high,0.101057,0.096769,0.111670


In [58]:
realized_vol = system_volatility(proxy, n_assets)

model_vol = {
    model: system_volatility(models[model], n_assets)
    for model in models
}

vol_df = pd.DataFrame({
    "Realized": realized_vol,
    **model_vol
}).dropna()

In [59]:
import plotly.graph_objects as go

fig = go.Figure()

# Realized volatility
fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["Realized"],
    mode="lines",
    name="Realized volatility",
    line=dict(width=2)
))

# Model forecasts
for model in models:
    
    fig.add_trace(go.Scatter(
        x=vol_df.index,
        y=vol_df[model],
        mode="lines",
        name=f"{model} forecast",
        line=dict(width=1)
    ))

# Low volatility threshold
fig.add_hline(
    y=low_threshold,
    line_dash="dash",
    line_color="green",
    annotation_text="Low volatility threshold",
    annotation_position="bottom right"
)

# High volatility threshold
fig.add_hline(
    y=high_threshold,
    line_dash="dash",
    line_color="red",
    annotation_text="High volatility threshold",
    annotation_position="top right"
)

fig.update_layout(
    title=f"System Volatility – Horizon h={H}",
    xaxis_title="Date",
    yaxis_title="Volatility",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

In [60]:
from IPython.display import display

print("============================================================")
print(f"FINAL EVALUATION SUMMARY (H = {H})")
print("============================================================")

print("\n1. Covariance Forecast Performance (Full Sample)")
display(cov_summary_table)

print("\n2. Variance and Correlation Forecast Performance (Full Sample)")
display(component_summary_table)

print("\n3. Average QLIKE by Volatility Regime")
display(qlike_regime_table)

print("\n4. Average Frobenius Loss by Volatility Regime")
display(frob_regime_table)

print("\n5. Variance MSE by Volatility Regime")
display(var_regime_table)

print("\n6. Correlation MSE by Volatility Regime")
display(corr_regime_table)

FINAL EVALUATION SUMMARY (H = 21)

1. Covariance Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_qlike,avg_frobenius,spd_fixes
model,,,,,,
EWMA,1235,2021-01-04,2025-12-02,5.235795,0.000002,0
DCC,1235,2021-01-04,2025-12-02,4.488577,0.000017,0
HAR,1235,2021-01-04,2025-12-02,9.260705,0.000002,8



2. Variance and Correlation Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
model,,,,,
EWMA,1235,2021-01-04,2025-12-02,1.491691e-07,0.080336
DCC,1235,2021-01-04,2025-12-02,2.108982e-06,0.075739
HAR,1235,2021-01-04,2025-12-02,1.826949e-07,0.091442



3. Average QLIKE by Volatility Regime


...

,EWMA,DCC,HAR
regime,,,
low,6.027697e-07,0.000048,4.997600e-07
mid,1.211043e-06,0.000008,1.215071e-06
high,3.213900e-06,0.000006,4.478937e-06



5. Variance MSE by Volatility Regime


,EWMA,DCC,HAR
regime,,,
low,5.092294e-08,5.976040e-06,4.005794e-08
mid,1.171445e-07,9.234530e-07,1.129261e-07
high,3.113609e-07,6.091462e-07,4.646436e-07



6. Correlation MSE by Volatility Regime


,EWMA,DCC,HAR
regime,,,
low,0.073199,0.075213,0.083123
mid,0.073534,0.065470,0.085478
high,0.101057,0.096769,0.111670
